
# Lalal.ai Downloader and Merger

This notebook provides a Python-based solution for downloading audio segments exported from Lalal.ai and merging them into a single audio file. It supports both vocals and no-vocals playlists and works seamlessly in Google Colab.

## Features
- **URL or File ID Input**: Accepts either a full URL or a file ID for processing.
- **Flexible Playlist Selection**: Allows users to choose between vocals or no-vocals playlists.
- **Automated Download**: Sequentially downloads all available segments.
- **Audio Merging**: Combines downloaded segments into a single MP3 file.
- **Colab Integration**: Designed for use in Google Colab with built-in file download support.

## Prerequisites
Before running the notebook, install the necessary dependencies:


In [1]:
!pip install pydub requests audioop-lts


## How to Use
1. **Extract File ID**:
   - Open the developer console in your browser (e.g., press `F12` in Chrome).
   - Go to the **Network** tab and play the converted song.
   - Locate the segment URLs and right click, copy and copy the URL.

2. **Run the Notebook**:
   - Input either a full URL or the extracted file ID when prompted.

3. **Choose Playlist**:
   - Specify whether you want the vocals (`yes`) or no-vocals (`no`) playlist.

4. **Download and Merge**:
   - The script will download all segments and merge them into `merged_audio.mp3`.
   - The merged file will be available for download directly from Colab.

## Example Input
- URL:  
  `https://d.lalal.ai/media/split/298a10ab-c995-488b-899c-3c81e2118c9c/ebb93a82-85ba-4f9f-b8d1-2b675e021bef/vocals_playlist/segment-000.mp3`
- File ID:  
  `298a10ab-c995-488b-899c-3c81e2118c9c/ebb93a82-85ba-4f9f-b8d1-2b675e021bef`

## Output
- Individual segments downloaded to the `segments` directory.
- Merged audio saved as `merged_audio.mp3`.

## Notes
- Ensure the song is fully converted on Lalal.ai to access all segments. The script won't work on preview segments.
- The script automatically downloads and merges all available segments.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning) # Ocultar advertencias de pydub

import requests
from pydub import AudioSegment
import os
from google.colab import files
import re
import shutil

def extract_info(input_str):
    """Extrae la URL base y el nombre de la capa (ej. vocals@0_playlist)."""
    # Busca el patrón exacto para extraer la capa (vocals@0, vocals@1, no_vocals, etc.)
    match_segment = re.match(r"^(.*?/([^/]+))/segment-\d+\.mp3", input_str)
    if match_segment:
        base_path = match_segment.group(1)   # ej: https://.../vocals@0_playlist
        layer_name = match_segment.group(2)  # ej: vocals@0_playlist
        return base_path + "/segment-", layer_name

    return None, None

def download_segment(base_url, segment_number, download_dir):
    """Descarga un segmento individual."""
    url = f"{base_url}{segment_number:03}.mp3"
    file_path = os.path.join(download_dir, f"segment-{segment_number:03}.mp3")

    try:
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024):
                    file.write(chunk)
            print(f"Descargado: segmento {segment_number:03}")
            return file_path
        else:
            if segment_number == 0:
                print(f"Segmento 0 no encontrado. Código: {response.status_code}")
            return None
    except Exception as e:
        print(f"Error descargando el segmento {segment_number}: {e}")
        return None

def merge_segments(segment_files, output_filename):
    """Une los segmentos en un solo archivo mp3."""
    print("\nUniendo segmentos... Esto puede tomar un momento.")
    merged = AudioSegment.empty()

    for file in segment_files:
        audio = AudioSegment.from_file(file)
        merged += audio

    merged.export(output_filename, format="mp3")
    print(f"✅ Audio guardado exitosamente como: {output_filename}")
    files.download(output_filename)

def main():
    input_str = input("Ingresa el link de un segmento (ej: .../vocals@0_playlist/segment-018.mp3): ").strip()

    base_url, layer_name = extract_info(input_str)

    if not base_url:
        print("❌ URL inválida. Asegúrate de pegar el enlace completo.")
        return

    print(f"\n🎵 Capa detectada: {layer_name}")

    # Crear carpeta específica para esta capa (evita mezclar audios de diferentes links)
    download_dir = os.path.join("segments", layer_name)
    os.makedirs(download_dir, exist_ok=True)

    segment_files = []
    segment_number = 0
    missing_count = 0

    print("Iniciando descargas...")
    while True:
        file_path = download_segment(base_url, segment_number, download_dir)
        if file_path:
            segment_files.append(file_path)
            missing_count = 0
        else:
            missing_count += 1
            # Si el segmento 0 no existe, probamos con el 1
            if segment_number == 0 and missing_count == 1:
                pass
            else:
                break # Fin de los segmentos
        segment_number += 1

    if segment_files:
        # El archivo final tendrá el nombre exacto de la capa para no confundirlos
        output_filename = f"{layer_name}_completo.mp3"
        merge_segments(segment_files, output_filename)

        # Borrar los fragmentos temporales de esta capa para limpiar Colab
        try:
            shutil.rmtree(download_dir)
        except:
            pass
    else:
        print("❌ No se descargó nada. Verifica que el link no haya caducado (Lalal.ai caduca los links rápido).")

if __name__ == "__main__":
    main()